In [ ]:
!pip install fasttext numpy==1.26.4 pandas pymorphy3 tqdm

In [ ]:
import fasttext.util

In [ ]:
fasttext.util.download_model('ru', if_exists='ignore')

In [ ]:
pretrained_ft = fasttext.load_model('cc.ru.300.bin')

In [ ]:
import numpy as np
import pandas as pd
from google.colab import files
from pymorphy3 import MorphAnalyzer
import io
from tqdm import tqdm

In [ ]:
def prepare_data(file_path):
    df = pd.read_csv(file_path)
    df["text"] = "__label__" + df["LFFUNC"] + "_" + df["LFVAL"] + " " + df["LFARG"]
    df["text"].to_csv("fasttext_train.txt", index=False, header=False)

    func_pos_dict = {}
    for func in df["LFFUNC"].unique():
        pos_set = set(df[df["LFFUNC"] == func]["POS_val"].dropna().unique())
        func_pos_dict[func] = pos_set

    return df, func_pos_dict
    return df

In [ ]:
def train_model():
    model = fasttext.train_supervised(
        input="fasttext_train.txt",
        lr=0.1,
        epoch=10,
        wordNgrams=2,
        dim=100,
        verbose=2
    )
    return model

In [ ]:
def get_word_pos(word):
    morph = MorphAnalyzer()
    parsed = morph.parse(word)[0]
    return parsed.tag.POS

In [ ]:
def predict_lf(model, function, word, func_pos_dict, top_k=5, expand_topn=3):
    global pretrained_ft

    labels, probs = model.predict(word, k=1000)
    results = []

    allowed_pos = func_pos_dict.get(function, set())

    for label, prob in zip(labels, probs):
        if f"__label__{function}_" in label:
            lf_val = label.replace(f"__label__{function}_", "").lower()
            first_word = lf_val.split()[0]
            predicted_pos = get_word_pos(first_word)

            if allowed_pos and predicted_pos not in allowed_pos:
                continue
            results.append((lf_val, prob, 'original'))

    results = sorted(results, key=lambda x: -x[1])

    if pretrained_ft is not None and expand_topn > 0:
        expanded = []
        used_words = set(r[0].split()[0].lower() for r in results)

        for val, prob, _ in results:
            first_word = val.split()[0]
            similar = pretrained_ft.get_nearest_neighbors(first_word, k=expand_topn)
            for sim_score, sim_word in similar:
                sim_word = sim_word.lower()
                if sim_word in used_words:
                    continue
                sim_pos = get_word_pos(sim_word)
                if allowed_pos and sim_pos not in allowed_pos:
                    continue
                combined_prob = prob * sim_score
                expanded.append((sim_word, combined_prob, 'similar'))
                used_words.add(sim_word)

        results.extend(expanded)

    results = sorted(results, key=lambda x: -x[1])[:top_k]

    return [(val, prob) for val, prob, _ in results]

In [ ]:
test_cases_old = [
    ('MAGN', 'довод'),
    ('OPER1', 'домино'),
    ('OPER2', 'арест'),
    ('INCEPOPER1', 'азарт'),
    ('FUNC0', 'дорога'),
    ('INCEPFUNC0', 'день'),
    ('CAUSFUNC0', 'встреча'),
    ('REAL1', 'газета'),
    ('REAL1-M', 'долг')
]

test_cases_new = [
    ('LOC', 'дом'),
    ('OPER1', 'оценка'),
    ('MAGN', 'друг'),
    ('ADV2-UN', 'причина'),
    ('ADV1-UN', 'надежда'),
    ('CAUSFUNC0', 'заседание'),
    ('FUNC0', 'дело'),
    ('INCEPOPER1', 'право'),
    ('OPER2', 'внимание')
]

test_cases = test_cases_old + test_cases_new

In [ ]:
file_name = "lf_data_words_with_pos.csv"

df, func_pos_dict = prepare_data(file_name)
model = train_model()

for func, arg in test_cases:
    print(f"\n{func}({arg}):")
    predictions = predict_lf(model, func, arg, func_pos_dict)

    if not predictions:
        print("Нет предсказаний")
    else:
        for i, (val, prob) in enumerate(predictions, 1):
            first_word = val.split()[0]
            predicted_pos = get_word_pos(first_word)
            print(f"{i}. {val} ({prob:.4f}, POS: {predicted_pos})")